In [0]:
import importlib
import sys
from pathlib import Path

project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.storage_config as storage_config
from pyspark.sql import functions as F
importlib.reload(storage_config)
storage_config.spark = spark
storage_config.configure_storage()

In [0]:
df_sellers = (
    spark.read
    .option("header","True")
    .option("inferSchema", "True")
    .csv("abfss://raw-data@secondstorage89.dfs.core.windows.net/sellers/")
)

In [0]:
df = (
    df_sellers
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
df.show(10)

In [0]:
df.write.mode("overwrite").save("abfss://bronze@secondstorage89.dfs.core.windows.net/sellers/")